# Collocation: The Company a Word Keeps

**Computational Text Analysis Pack · Module 05 — Concordance and Collocation**

*Technique:* Window co-occurrence; span and its effect on results

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/03-comparing-corpora/05b_collocation.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

*"You shall know a word by the company it keeps"* — Firth, 1957, and still the
best one-line statement of the idea.

A **collocate** is a word that appears near your keyword more often than chance
would put it there. Collocation is how you characterise a word's meaning in a
corpus without defining it, and it is the method behind every word-embedding
model you have heard of.

The single decision that shapes everything: the **span**, how many words either
side count as "near".

## The tooling

Standard library again — concordancing and collocation are slicing and counting.
**AntConc** and **CQPweb** are the dedicated tools; **#LancsBox**'s GraphColl
does the network version in module 19.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
def collocates(texts, term, span=5, min_count=5):
    """Words occurring within `span` tokens either side of `term`."""
    near, total = Counter(), Counter()
    term = term.lower()
    hits = 0
    for t in texts:
        toks = tokenise(t)
        total.update(toks)
        for i, w in enumerate(toks):
            if w == term:
                hits += 1
                lo, hi = max(0, i - span), min(len(toks), i + span + 1)
                near.update(toks[lo:i] + toks[i + 1:hi])
    near = {w: n for w, n in near.items() if n >= min_count}
    return hits, near, total

In [ ]:
TERM = "native"             # ← change this
SPAN = 5                    # ← and this

hits, near, total = collocates(df_colonial["text"], TERM, span=SPAN)
print(f"'{TERM}' occurs {hits:,} times; {len(near):,} words co-occur within ±{SPAN}")
print()
print("Raw co-occurrence counts:")
for w, n in sorted(near.items(), key=lambda kv: -kv[1])[:15]:
    print(f"   {w:<18}{n:>6}")

## Raw counts are not collocation

That list is *the*, *of*, *and* — the commonest words in the language, which
appear near everything. Raw co-occurrence measures nothing but frequency.

A collocate is a word that appears near the keyword **more than its overall rate
predicts**. That needs the same expected-versus-observed reasoning as keyness.

In [ ]:
def collocation_strength(hits, near, total, span=5, min_count=5):
    """Observed co-occurrence against what the word's overall rate predicts."""
    n_tokens = sum(total.values())
    window = hits * span * 2
    rows = []
    for w, obs in near.items():
        if obs < min_count:
            continue
        expected = window * total[w] / n_tokens
        if expected <= 0:
            continue
        rows.append({"word": w, "observed": obs, "expected": expected,
                     "ratio": obs / expected,
                     "corpus_count": total[w]})
    out = pd.DataFrame(rows)
    return out.sort_values("ratio", ascending=False).reset_index(drop=True)

strength = collocation_strength(hits, near, total, span=SPAN, min_count=8)
print(f"Strongest collocates of '{TERM}' (±{SPAN}):")
print(strength.head(20).to_string(index=False, float_format=lambda v: f"{v:.2f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
top = strength.head(15).iloc[::-1]
ax.barh(top["word"], top["ratio"])
ax.axvline(1, color="grey", lw=0.8)
ax.set_xlabel(f"times more often near '{TERM}' than chance predicts")
ax.set_title(f"Collocates of '{TERM}' in the colonial corpus (±{SPAN})")
plt.tight_layout(); plt.show()

### 🔧 Your turn 1

Set `SPAN` to 2, then 10, and re-run.

A span of 2 picks up grammatical partners — the adjective-noun pairs. A span of
10 picks up subject matter — words from the same discussion. Neither is right;
they answer different questions, and corpus linguists conventionally report ±5.

## The rare-word problem

Sort by ratio alone and the top is full of words appearing five or six times. The
ratio is huge and unstable — the same Zipf problem as everywhere else.

In [ ]:
print("Sorted by ratio, no frequency floor — the unstable version:")
loose = collocation_strength(hits, near, total, span=SPAN, min_count=3)
print(loose.head(8).to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print()
print("With a floor of 25 occurrences — the version worth reporting:")
firm = strength[strength["observed"] >= 25]
print(firm.head(12).to_string(index=False, float_format=lambda v: f"{v:.2f}"))

### 🔧 Your turn 2

Run the whole notebook on `df` (contemporary news) with a word you research,
rather than on the colonial corpus.

Then run KWIC (05a) on the two or three strongest collocates. Collocation tells
you what to look at; the concordance tells you what it means.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** At ±2 you see the immediate grammar — for *native* that is
*chief*, *races*, *quarter*, the nouns it modifies. At ±10 you see the discourse
it belongs to: *government*, *labour*, *administration*. The narrow span is
evidence about the phrase; the wide span is evidence about the topic. Say which
you used, because a reader cannot reconstruct it from the results.

**Your turn 2.** The pairing is the point of this module. Collocation is a
ranking device over a corpus too large to read; KWIC is the reading. A paper that
reports collocates without ever quoting a line of context is asking the reader to
trust a number about meaning — and a paper that quotes lines without saying how
they were selected is asking them to trust your sampling. Do both.

</details>

## Where this stops

You can measure which words cluster around a keyword. Next: **05c**, which puts a
proper statistic on that measure and shows what each choice of statistic favours.